In [ ]:
import parcels
import copernicusmarine
import xarray as xr
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
copernicus_args = {
    "start_datetime": "2025-06-10",
    "end_datetime": "2025-06-12",
    "minimum_longitude": -49,
    "maximum_longitude": -42,
    "minimum_latitude": 18,
    "maximum_latitude": 26,
    "service": "arco-geo-series",
    "chunk_size_limit": 1,
}

ds_uv = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
    variables=["uo", "vo"],  # TODO add vertical velocity
    **copernicus_args,
)

ds_w = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy-wcur_anfc_0.083deg_P1D-m",
    variables=["wo"],
    **copernicus_args,
)

ds_s = copernicusmarine.open_dataset(
  dataset_id="cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m",
  variables=["so"],
  **copernicus_args,
)

ds_t = copernicusmarine.open_dataset(
  dataset_id="cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m",
  variables=["thetao"],
  **copernicus_args,
)

# combine into one dataset
ds = parcels.convert.copernicusmarine_to_sgrid(
    fields={
        "U": ds_uv["uo"],
        "V": ds_uv["vo"],
        "W": ds_w["wo"],
        "S": ds_s["so"],
        "T": ds_t["thetao"],
    }
)

# explicitly load dataset
ds.load()
display(ds)

# convert to parcels fieldset
fieldset = parcels.FieldSet.from_sgrid_conventions(ds)
fieldset.to_chunk_cached_arrays()
# fieldset.describe()  # DEBUG very slow

In [ ]:
# This is where the science happens - how to define rise velocity?
def Rising(particles, fieldset):
    risevel = 0.10  # Example rise velocity in m/s
    particles.dz -= risevel * particles.dt

In [ ]:

pset = parcels.ParticleSet(fieldset, x=-46.15, y=22.84, z=2000)
oufile = parcels.ParticleFile(
    "output.parquet",
    outputdt=np.timedelta64(15, "m"),
    mode="w",
)

pset.execute(
    [parcels.kernels.AdvectionRK2_3D, Rising],
    dt=np.timedelta64(15, "m"),
    runtime=np.timedelta64(3, "D"),
    output_file=oufile,
)


In [ ]:
df = parcels.read_particlefile("output.parquet")

plt.plot(df["t"], df["z"], '.-')
plt.show()

plt.plot(df["x"], df["y"], '.-')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(13, 8))
ax = plt.axes(projection="3d")
ax.view_init(azim=-145)
ax.plot3D(df["x"], df["y"], df["z"], '.-', color="gray")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_zlabel("Depth (m)")
ax.set_zlim(df["z"].max(), 0)


In [ ]:
plt.plot(np.diff(df["z"]))
plt.show()